# nano-gpt on an NVIDIA GPU (Colab launcher)

This notebook only calls the repository's own scripts. It contains no model or training code.

Before running: **Runtime > Change runtime type > GPU**.

The Colab filesystem is temporary. The last cells zip the run directory for download; nothing is kept once the runtime is recycled.

## 1. GPU and software

In [ ]:
!nvidia-smi

In [ ]:
import sys
import torch

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))
TORCH_BEFORE_INSTALL = torch.__version__

## 2. Clone the repository and install dependencies

Set `REF` to a commit hash for a reproducible run, or to a branch name such as `main` for the latest code.

`requirements.txt` only asks for `torch>=2.2.0`, which Colab's preinstalled CUDA build already satisfies, so pip keeps it. The last line of the cell checks this.

In [ ]:
import os
import time

REPO_URL = "https://github.com/AdebanjiAdelowo/nano-gpt.git"
REF = "main"
REPO = "/content/nano-gpt"

# One directory per notebook session, so re-running never overwrites an earlier run.
STAMP = time.strftime("%Y%m%d-%H%M%S")
SMOKE_DIR = f"results/colab-smoke-{STAMP}"
RUN_DIR = f"results/colab-cuda-{STAMP}"

if not os.path.exists(REPO):
    !git clone {REPO_URL} {REPO}
%cd {REPO}
!git fetch -q --all && git checkout -q {REF}
!git log -1 --format='%H %ad %s' --date=short
!pip install -q -r requirements.txt
!python -c "import torch; print('PyTorch after install:', torch.__version__, '| CUDA available:', torch.cuda.is_available())"
print("PyTorch before install:", TORCH_BEFORE_INSTALL)

## 3. Short CUDA smoke training (50 steps)

`--device cuda` fails with a clear error if CUDA is not available; it never falls back to CPU.

In [ ]:
!python train.py --device cuda --max-iters 50 --eval-interval 25 --eval-iters 10 --out-dir {SMOKE_DIR}

In [ ]:
# Gate: run_meta.json is written only after training finishes, so its presence means the smoke run succeeded.
import json
import math

smoke_meta_path = f"{SMOKE_DIR}/run_meta.json"
assert os.path.exists(smoke_meta_path), "Smoke test failed: read the error above before running the full training"
with open(smoke_meta_path) as f:
    smoke = json.load(f)
assert smoke["resolved_device"] == "cuda", smoke["resolved_device"]
assert math.isfinite(smoke["results"]["final_val_loss"]), "non-finite loss"
SMOKE_PASSED = True
print("Smoke test passed on", smoke["gpu_name"], "| val loss", round(smoke["results"]["final_val_loss"], 4))

## 4. Normal CUDA training (default configuration, 3000 steps)

In [ ]:
assert globals().get("SMOKE_PASSED"), "Run the smoke test and its check first"
!python train.py --device cuda --out-dir {RUN_DIR}

## 5. Generate text from the new checkpoint

In [ ]:
assert os.path.exists(f"{RUN_DIR}/run_meta.json"), "Full training did not finish: read the error above"
!python generate.py --device cuda --checkpoint {RUN_DIR}/checkpoint.pt --max_new_tokens 300 --seed 0

## 6. Loss curve and run metadata

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{RUN_DIR}/loss_curve.png"))

In [ ]:
with open(f"{RUN_DIR}/run_meta.json") as f:
    print(json.dumps(json.load(f), indent=2))

## 7. Keep the results

Download the archive now, or copy it to Google Drive. It holds the checkpoints, loss curves and run metadata of both runs.

In [ ]:
ARCHIVE = f"/content/nano-gpt-{os.path.basename(RUN_DIR)}.zip"
!zip -r -q {ARCHIVE} {RUN_DIR} {SMOKE_DIR}
!ls -lh {ARCHIVE}

from google.colab import files
files.download(ARCHIVE)

In [ ]:
# Optional: copy to Google Drive instead of downloading.
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/nano-gpt-runs && cp {ARCHIVE} /content/drive/MyDrive/nano-gpt-runs/